# L3–L4f and L3–L4m CHRR comparisons

Compare saved CHRR samples separately for female and male L4 models.
The fixed exclusions, minimum sampled range of 0.001 model flux units,
KS statistic and relative median-difference ranking are retained.
P and adjusted q values are descriptive and do not select reactions.


In [ ]:
from scipy.stats import false_discovery_control
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from IPython.display import display

INPUT_DIR = Path("inputs")
RESULT_DIR = Path("outputs")
OUTPUT_DIR = RESULT_DIR / "comparison_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXCLUDED = [
    "MAR00023",
    "MAR01506",
    "MAR01507",
    "MAR02960",
    "MAR02993",
    "MAR03031",
    "MAR03036",
    "MAR03041",
    "MAR03131",
    "MAR06630",
    "MAR06631",
    "MAR06632",
    "MAR06633",
    "MAR06634",
    "MAR06635",
    "MAR06636",
    "MAR06637",
    "MAR06638",
    "MAR06639",
    "MAR06641",
    "MAR06642",
    "MAR06646",
    "MAR06647",
    "MAR06648",
    "MAR06649",
    "MAR06651",
    "MAR06652",
    "MAR06653",
    "MAR06654",
    "MAR06655",
    "MAR06656",
    "MAR06657",
    "MAR06658",
    "MAR06659",
    "MAR06660",
    "MAR06662",
    "MAR06663",
    "MAR06664",
    "MAR06665",
    "MAR06666",
    "MAR06667",
    "MAR06668",
    "MAR06669",
    "MAR06670",
    "MAR06671",
    "MAR06672",
    "MAR06673",
    "MAR06674",
    "MAR06675",
    "MAR06676",
    "MAR06677",
    "MAR06678",
    "MAR06680",
    "MAR06682",
    "MAR06685",
    "MAR06686",
    "MAR06687",
    "MAR06690",
    "MAR06691",
    "MAR06692",
    "MAR06693",
    "MAR06694",
    "MAR06695",
    "MAR06697",
    "MAR06699",
    "MAR06700",
    "MAR06701",
    "MAR06702",
    "MAR06703",
    "MAR06704",
    "MAR06705",
    "MAR08697",
    "MAR08698",
    "MAR08699",
    "MAR08702",
    "MAR08703",
    "MAR08704",
    "MAR08706",
    "MAR08709",
    "MAR08710",
    "MAR08711",
    "MAR08712",
    "MAR08716",
    "MAR11894",
    "MAR11895",
    "MAR11896",
    "MAR11897",
    "MAR20001",
    "MAR20003",
    "MAR20005",
    "MAR20006",
    "MAR20007",
    "MAR20008",
]


def drop_cols(frame):
    """Remove recorded stage/sex IDs from a reactions-as-columns frame."""
    return frame.drop(columns=[rid for rid in EXCLUDED if rid in frame.columns])


CHRR = INPUT_DIR / "CHRR_sampling"
PCA = CHRR / "pca"
RANGE_MIN = 0.001
SIGN_TOL = 1e-09
SAMPLE_PATHS = {
    "L3": PCA / "L3_c_samples.csv",
    "L4f": PCA / "L4f_c_samples.csv",
    "L4m": PCA / "L4m_c_samples.csv",
}
COMPARATORS = ["L4f", "L4m"]


def save(data, name):
    d = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    d.to_csv(OUTPUT_DIR / name, sep="\t", index=False)
    return d


samples = {}
sample_inventory = []
for stage, path in SAMPLE_PATHS.items():
    raw = pd.read_csv(path)
    samples[stage] = drop_cols(raw)
    sample_inventory.append(
        dict(
            model=stage,
            samples=len(raw),
            sample_reactions_before_exclusion=raw.shape[1],
            sample_reactions_after_exclusion=samples[stage].shape[1],
            excluded_reaction_ids=";".join(
                sorted(set(raw.columns) - set(samples[stage].columns))
            ),
        )
    )
ev = (
    pd.read_csv(PCA / "reaction_evidence_table.csv")
    .drop_duplicates("rxnID")
    .set_index("rxnID")
)
ranges = {k: d.max() - d.min() for k, d in samples.items()}
save(sample_inventory, "sample_inventory.tsv")




def shift_sign(x):
    return (
        "comparator_higher"
        if x > SIGN_TOL
        else "L3_higher" if x < -SIGN_TOL else "numerically_tied"
    )


def annotate(df):
    for column in ["probable_name", "subSystem", "equation"]:
        df[column] = df.rxnID.map(ev[column])
    return df


In [ ]:
results = {}
eligibility = []
comparison_rows = []
a = samples["L3"]
for other in COMPARATORS:
    b = samples[other]
    shared = sorted(set(a.columns) & set(b.columns))
    eligible = []
    for rid in shared:
        ok = ranges["L3"][rid] >= RANGE_MIN and ranges[other][rid] >= RANGE_MIN
        eligibility.append(
            dict(
                comparison="L3_vs_" + other,
                reaction_id=rid,
                range_L3=ranges["L3"][rid],
                range_comparator=ranges[other][rid],
                eligible=ok,
            )
        )
        if ok:
            eligible.append(rid)
    rows = []
    for rid in eligible:
        D, p = ks_2samp(a[rid].values, b[rid].values)
        m1 = a[rid].median()
        m2 = b[rid].median()
        span = max(a[rid].max(), b[rid].max()) - min(a[rid].min(), b[rid].min())
        rows.append((rid, D, p, m1, m2, m2 - m1, span))
    d = pd.DataFrame(
        rows,
        columns=[
            "rxnID",
            "D",
            "pvalue",
            "median_L3",
            "median_comparator",
            "delta_median",
            "pooled_range",
        ],
    )
    d["CHRR_min_L3"] = [a[rid].min() for rid in d.rxnID]
    d["CHRR_max_L3"] = [a[rid].max() for rid in d.rxnID]
    d["CHRR_min_comparator"] = [b[rid].min() for rid in d.rxnID]
    d["CHRR_max_comparator"] = [b[rid].max() for rid in d.rxnID]
    d["absolute_delta_median"] = d.delta_median.abs()
    d["qvalue"] = false_discovery_control(d.pvalue)
    d["relative_delta"] = d.delta_median.abs() / d.pooled_range
    d["rank_D"] = d.D.rank(ascending=False, method="min").astype(int)
    d["rank_relative_delta"] = d.relative_delta.rank(
        ascending=False, method="min"
    ).astype(int)
    d["combined_score"] = d.rank_D + d.rank_relative_delta
    d = d.sort_values("combined_score").reset_index(drop=True)
    d["median_shift_direction"] = d.delta_median.map(shift_sign)
    d["comparison"] = "L3_vs_" + other
    results[other] = annotate(d)
    save(results[other], f"pairwise_ks_L3_vs_{other}.tsv")
    comparison_rows.append(
        dict(
            comparison="L3_vs_" + other,
            samples_L3=len(a),
            samples_comparator=len(b),
            shared_after_original_exclusions=len(shared),
            tested_reactions=len(d),
            excluded_by_original_range_filter=len(shared) - len(d),
            D_equal_1=int(d.D.eq(1).sum()),
            median_D=d.D.median(),
            median_relative_delta=d.relative_delta.median(),
            comparator_higher=int(
                d.median_shift_direction.eq("comparator_higher").sum()
            ),
            L3_higher=int(d.median_shift_direction.eq("L3_higher").sum()),
            numerically_tied=int(d.median_shift_direction.eq("numerically_tied").sum()),
        )
    )
save(eligibility, "all_shared_reaction_eligibility.tsv")
comparison_summary = save(comparison_rows, "comparison_summary.tsv")
display(comparison_summary)


In [ ]:
combined = pd.concat([results[stage] for stage in COMPARATORS], ignore_index=True)
save(combined, "L3_L4_sex_comparisons.tsv")
display(combined)
